In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd

# 1. 讀取訓練資料 (路徑請確保與你上一個步驟印出的路徑一致)
train_path = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip'
df_train = pd.read_csv(train_path)

# 2. 顯示前五筆資料，看看真實的留言長怎樣
print("=== 前五筆訓練資料 ===")
display(df_train.head())

# 3. 統計各個惡意標籤的數量與比例 (作業要求的重點！)
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print("\n=== 各標籤的正樣本數量與比例 ===")
for label in labels:
    count = df_train[label].sum()
    ratio = (count / len(df_train)) * 100
    print(f"{label:<15}: {count:>6} 筆 ({ratio:.2f}%)")

=== 前五筆訓練資料 ===


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0



=== 各標籤的正樣本數量與比例 ===
toxic          :  15294 筆 (9.58%)
severe_toxic   :   1595 筆 (1.00%)
obscene        :   8449 筆 (5.29%)
threat         :    478 筆 (0.30%)
insult         :   7877 筆 (4.94%)
identity_hate  :   1405 筆 (0.88%)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. 切分訓練集與驗證集 (80% 訓練, 20% 拿來當模擬考驗證)
print("正在切分資料...")
train_text, val_text, train_y, val_y = train_test_split(
    df_train['comment_text'], df_train[labels], test_size=0.2, random_state=42
)

# 2. 將文字轉換成數字向量 (TF-IDF)
print("正在將文字轉換成 TF-IDF 向量 (這可能需要一兩分鐘，請稍候)...")
# 為了加快速度和避免記憶體爆炸，我們只取最常出現的 10,000 個字，並排除英文的無意義停用詞
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english') 
X_train = vectorizer.fit_transform(train_text)
X_val = vectorizer.transform(val_text)

# 3. 訓練基準模型並計算 AUC 分數
print("開始訓練邏輯斯迴歸模型...\n")
val_aucs = []
for label in labels:
    # 針對這 6 個標籤，我們各訓練一個分類器
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, train_y[label])
    
    # 預測驗證集的「惡意機率」 (predict_proba 會回傳兩個機率，我們取 [:, 1] 正樣本機率)
    val_preds = model.predict_proba(X_val)[:, 1]
    
    # 計算並印出 AUC 分數
    auc = roc_auc_score(val_y[label], val_preds)
    val_aucs.append(auc)
    print(f"{label:<15} 驗證集 AUC: {auc:.4f}")

print("-" * 30)
print(f"👉 基準模型平均 AUC (Baseline Score): {sum(val_aucs)/len(val_aucs):.4f}")

正在切分資料...
正在將文字轉換成 TF-IDF 向量 (這可能需要一兩分鐘，請稍候)...
開始訓練邏輯斯迴歸模型...

toxic           驗證集 AUC: 0.9667
severe_toxic    驗證集 AUC: 0.9798
obscene         驗證集 AUC: 0.9827
threat          驗證集 AUC: 0.9857
insult          驗證集 AUC: 0.9739
identity_hate   驗證集 AUC: 0.9701
------------------------------
👉 基準模型平均 AUC (Baseline Score): 0.9765


In [5]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

# 🚨 檢查點：確認你的 GPU 已經成功開啟
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"目前使用的硬體裝置為: {device} (如果顯示 cpu，請務必去右側面板把 Accelerator 改成 GPU！)")

# 1. 重新整理 Hugging Face 需要的資料格式
# 把文字和我們那 6 個標籤答案打包在一起
train_df_hf = pd.DataFrame({'text': train_text})
train_df_hf['labels'] = train_y[labels].values.astype(float).tolist()

val_df_hf = pd.DataFrame({'text': val_text})
val_df_hf['labels'] = val_y[labels].values.astype(float).tolist()

# 轉換成 Hugging Face 專用的 Dataset 物件
hf_train_dataset = Dataset.from_pandas(train_df_hf)
hf_val_dataset = Dataset.from_pandas(val_df_hf)

# 2. 載入 DistilBERT 的文字斷詞器 (Tokenizer)
print("正在下載並載入 DistilBERT Tokenizer...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 定義轉換函數 (照作業要求：限制最大長度 max_length=192)
def tokenize_function(examples):
    return tokenizer(examples["text"],padding="max_length", truncation=True, max_length=192)

print("正在對訓練集與驗證集進行文字編碼...")
tokenized_train = hf_train_dataset.map(tokenize_function, batched=True)
tokenized_val = hf_val_dataset.map(tokenize_function, batched=True)

print("🎉 文字編碼完成！我們準備好可以餵給 AI 讀書了。")

目前使用的硬體裝置為: cuda (如果顯示 cpu，請務必去右側面板把 Accelerator 改成 GPU！)
正在下載並載入 DistilBERT Tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

正在對訓練集與驗證集進行文字編碼...


Map:   0%|          | 0/127656 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

🎉 文字編碼完成！我們準備好可以餵給 AI 讀書了。


In [6]:
import os
import wandb

# 直接把密碼塞進環境變數，這樣它就不會跳出框框問你了！
os.environ["WANDB_API_KEY"] = "YOUR_WANDB_API_KEY_HERE"
wandb.login()
# 初始化一個專案名稱，你的實驗紀錄都會整理在這個專案下
wandb.init(project="toxic-comment-classification", name="distilbert-run-1")
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from scipy.special import expit
from sklearn.metrics import roc_auc_score

# 1. 載入 DistilBERT 模型大腦
# 因為我們有 6 個惡意標籤，所以要告訴模型 num_labels=6
# 並且指定這是一個「多標籤分類 (multi_label_classification)」問題
print("正在從網路上下載並初始化 DistilBERT 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=6,
    problem_type="multi_label_classification"
)

# 2. 定義「改考卷」的評分函數 (計算平均 AUC)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # 使用 sigmoid (expit) 把模型輸出的數字轉換成 0 到 1 之間的機率
    probs = expit(predictions)
    
    # 計算 6 個標籤的平均 AUC
    auc_scores = []
    for i in range(6):
        auc = roc_auc_score(labels[:, i], probs[:, i])
        auc_scores.append(auc)
    
    return {"macro_auc": np.mean(auc_scores)}

# 3. 設定特訓班的參數 (Training Arguments)
print("正在設定訓練參數...")
training_args = TrainingArguments(
    output_dir="./results",          # 訓練過程的暫存資料夾
    save_strategy="epoch",           # 每個 Epoch 存檔一次
    learning_rate=2e-5,              # 學習率 (學習步伐，不能太大也不能太小)
    per_device_train_batch_size=32,  # 一次餵 32 筆資料給 GPU 讀，避免記憶體爆炸
    per_device_eval_batch_size=32,   # 模擬考時一次讀 32 筆
    num_train_epochs=1,              # 🚨 先讓 AI 把 16 萬筆資料完整讀完「1 遍」就好 (省時間)
    weight_decay=0.01,               # 防止 AI 死背答案的懲罰機制
    load_best_model_at_end=True,     # 訓練結束後，自動幫我們挑選考最高分的那個模型
    metric_for_best_model="macro_auc", # 挑選標準就是看 AUC
    fp16=True,                       # 🚀 開啟半精度加速 (讓 GPU 跑快一倍的秘訣！)
    # 🌟 修正 W&B 沒圖表的關鍵設定，請直接補上這三行 🌟
    report_to="wandb",            # 1. 忘記寫這個，它就不會把分數傳給 W&B
    logging_steps=10,             # 2. 強制每 10 步就傳一次分數（這樣才看得到漂亮的折線圖！）
    eval_strategy="epoch"         # 3. 每個階段結束自動去算 eval/loss 驗證集分數
)

# 4. 把大腦、課本、考卷和參數通通丟進訓練器 (Trainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # 剛剛做好的訓練集 Token
    eval_dataset=tokenized_val,      # 剛剛做好的驗證集 Token
    compute_metrics=compute_metrics  # 剛剛寫好的改考卷函數
)

# 5. 💥 正式開火訓練！
print("\n🔥 特訓班正式開始！GPU 正在全速運轉中...")
trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 11430053 (11430053-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

正在設定訓練參數...

🔥 特訓班正式開始！GPU 正在全速運轉中...


Epoch,Training Loss,Validation Loss,Macro Auc
1,0.059843,0.037294,0.990769


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3990, training_loss=0.049333900803312625, metrics={'train_runtime': 608.0057, 'train_samples_per_second': 209.959, 'train_steps_per_second': 6.562, 'total_flos': 6341799196735488.0, 'train_loss': 0.049333900803312625, 'epoch': 1.0})

In [7]:
# 1. 讀取測試集資料 (請根據你實際的 test.csv 路徑修改)
test_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')

# 2. 將測試集的文字轉成列表
test_texts = test_df['comment_text'].fillna("").tolist()

# 3. 使用 Tokenizer 進行編碼
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=192)

# 4. 包裝成 Hugging Face 可以讀取的 Dataset 格式
class KaggleTestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item
    def __len__(self):
        return len(self.encodings.input_ids)

test_dataset_hf = KaggleTestDataset(test_encodings)
print("測試集編碼完成！")

測試集編碼完成！


In [8]:
import numpy as np

# 1. 讓模型進行預測
print("AI 正在寫考卷中，請稍候...")
test_predictions = trainer.predict(test_dataset_hf)

# 2. 使用 Sigmoid 函數將輸出結果轉換成 0~1 的機率值
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

pred_probs = sigmoid(test_predictions.predictions)
print("預測完成！")

AI 正在寫考卷中，請稍候...


預測完成！


In [9]:
# 1. 定義 6 個標籤名稱
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# 2. 建立預測結果的 DataFrame
submission_df = pd.DataFrame(pred_probs, columns=labels)

# 3. 在最左邊插入官方要求的 'id' 欄位
submission_df.insert(0, 'id', test_df['id'])

# 4. 匯出成 CSV 檔案
submission_df.to_csv('submission.csv', index=False)
print("🎉 submission.csv 匯出成功！你可以準備去右上角按 Save Version 提交囉！")

🎉 submission.csv 匯出成功！你可以準備去右上角按 Save Version 提交囉！


In [10]:
import wandb
wandb.finish()

eval/loss,▁
eval/macro_auc,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇███
+3,...


In [12]:
import wandb
wandb.init()  # 🌟 重新打開 W&B 的大門，讓 trainer 不要報錯
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. 預測驗證集
print("正在計算驗證集成績...")
val_preds = trainer.predict(tokenized_val)
y_true = val_preds.label_ids
y_pred = 1 / (1 + np.exp(-val_preds.predictions))

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# 2. 印出各類別獨立 AUC
print("\n🏆 各類別獨立 AUC:")
for i, label in enumerate(labels):
    try:
        auc = roc_auc_score(y_true[:, i], y_pred[:, i])
        print(f"- {label}: {auc:.5f}")
    except ValueError:
        print(f"- {label}: 無法計算")

# 3. 撈出錯最慘的 5 個句子
print("\n🔍 模型誤判範例:")
errors = np.abs(y_true - y_pred).mean(axis=1)
top_error_indices = errors.argsort()[-5:][::-1]

for i, idx in enumerate(top_error_indices):
    text = tokenizer.decode(tokenized_val[int(idx)]['input_ids'], skip_special_tokens=True)
    print(f"\n[誤判範例 {i+1}]")
    print(f"📝 內容: {text[:200]}...") 
    print(f"🎯 真實標籤: {y_true[idx]}")
    print(f"🤖 模型預測: {np.round(y_pred[idx], 3)}")

正在計算驗證集成績...

🏆 各類別獨立 AUC:
- toxic: 0.98805
- severe_toxic: 0.99127
- obscene: 0.99432
- threat: 0.99160
- insult: 0.98987
- identity_hate: 0.98951

🔍 模型誤判範例:

[誤判範例 1]
📝 內容: nig, nigs? nig, nigs? what about nig, nigs? i hear sometimes people call others those. nig, nigs?...
🎯 真實標籤: [0. 0. 1. 0. 1. 1.]
🤖 模型預測: [0.722 0.011 0.082 0.024 0.23  0.058]

[誤判範例 2]
📝 內容: { { | unblock | lick my hairy nuts you negroes and jews. } }...
🎯 真實標籤: [1. 1. 0. 0. 0. 0.]
🤖 模型預測: [0.956 0.249 0.75  0.162 0.817 0.746]

[誤判範例 3]
📝 內容: " just face the truth : you can ' t escape it! = = user : arilang1234 didn ' t edit your foul, slimy racist template, you loser. i suggest that you stop pretending that manchus are ethnically chinese....
🎯 真實標籤: [0. 0. 1. 0. 1. 1.]
🤖 模型預測: [0.655 0.003 0.063 0.004 0.306 0.026]

[誤判範例 4]
📝 內容: n ig ger nig ger hi n i g g e r...
🎯 真實標籤: [1. 1. 1. 0. 0. 1.]
🤖 模型預測: [0.601 0.061 0.22  0.084 0.176 0.115]

[誤判範例 5]
📝 內容: " you have evidence from me! i presented you with tone of evi